# 第 17 章：法律领域小模型项目

这个 notebook 对应 `lessons/17_legal_domain_project.md`，演示一个离线合同风险审查小闭环：脱敏合同条款、构造 SFT 样本、用 RAG 知识库生成结构化风险输出、校验 citation，并检查 model card 是否声明用途边界。

In [ ]:
from src.legal_contract_review.contract_review import (
    build_contract_sft_example,
    deidentify_contract_clause,
    review_contract_clause,
    validate_legal_citations,
    validate_legal_model_card,
)
from src.rag.baseline import (
    Document,
    HashingTextEmbedder,
    VectorStore,
    chunk_documents,
    citation_from_chunk,
)
from src.safety.governance import ModelCard

## 1. 合同条款脱敏

真实合同进入训练或日志前必须脱敏，同时保留金额、日期、主体角色等合同结构。

In [ ]:
raw_clause = (
    "甲方在2026年5月28日未按期交付，"
    "应赔偿乙方因此产生的一切损失，包括间接损失、可得利益损失及律师费。"
)
clause = deidentify_contract_clause(raw_clause)
print(clause)

## 2. 审查规范知识库

法律项目中的 RAG 知识库可以是合同模板、审查规范或已批准解释。

In [ ]:
documents = [
    Document(
        doc_id="guideline",
        title="合同审查规范",
        text="违约责任条款应关注责任范围、间接损失、可得利益损失、律师费和责任上限。",
    )
]
chunks = chunk_documents(documents, chunk_size=80)
store = VectorStore(chunks, HashingTextEmbedder(dim=64))
for chunk in chunks:
    print(chunk.chunk_id, chunk.text)

## 3. 结构化风险输出

教学版 review 函数用检索结果和规则生成 JSON 风险提示，不提供最终法律意见。

In [ ]:
review = review_contract_clause(clause, store, top_k=1, min_score=0.1)
print(review.to_json())

citations = [citation_from_chunk(chunks[0])]
validate_legal_citations(review, citations)

## 4. 生成 SFT 样本

通过校验的结构化输出可以写入合同 SFT 数据，训练模型学习固定输出格式和人工复核边界。

In [ ]:
sft_example = build_contract_sft_example(
    example_id="contract_sft_001",
    source_id="contract_doc_001",
    clause=raw_clause,
    assistant_output=review,
    risk_tags=["liability", "needs_human_review"],
)

sft_example.to_dict()

## 5. 无证据 unknown 路径

资料不足时，模型应输出 `unknown` 并触发人工复核，而不是编造法律依据。

In [ ]:
unknown = review_contract_clause("量子芯片制造步骤由PARTY_A负责。", store, top_k=1, min_score=0.95)
unknown.to_dict()

## 6. Model Card 边界

法律项目的 model card 必须明确用途限制和人工复核要求。

In [ ]:
model_card = ModelCard(
    model_name="legal-contract-review",
    version="v1",
    base_model="tiny-base",
    intended_use=["合同条款风险提示"],
    out_of_scope_use=["不提供最终法律意见", "不替代律师"],
    training_data="deidentified_contract_sft_v1",
    evaluation="eval_report.md",
    limitations=["单条条款不足以形成完整法律判断"],
    safety=["高风险条款必须人工复核"],
    deployment="local teaching demo",
    owner="course-maintainer",
)

validate_legal_model_card(model_card)
model_card.to_dict()